# Notebook 1 — Exploratory Data Analysis
**Nifty 100 Financial Intelligence Pipeline**

Understand the data before building anything. Produces 20+ visualisations across revenue distributions, sector aggregations, correlation matrices, null heatmaps, and outlier analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

DATA = Path('../data/clean')
pl  = pd.read_csv(DATA / 'profitandloss.csv')
bs  = pd.read_csv(DATA / 'balancesheet.csv')
cf  = pd.read_csv(DATA / 'cashflow.csv')
co  = pd.read_csv(DATA / 'companies.csv')
an  = pd.read_csv(DATA / 'analysis.csv')

# Use non-TTM annual rows only
pl  = pl[pl['is_ttm'] == False].copy()
bs  = bs[bs['is_ttm'] == False].copy()
cf  = cf[cf['is_ttm'] == False].copy()

# Latest-year snapshot per company
pl_latest = pl.sort_values('fiscal_year').groupby('company_id').last().reset_index()
bs_latest = bs.sort_values('fiscal_year').groupby('company_id').last().reset_index()
cf_latest = cf.sort_values('fiscal_year').groupby('company_id').last().reset_index()

print(f'Companies: {co.shape[0]}')
print(f'P&L rows:  {pl.shape[0]}')
print(f'BS rows:   {bs.shape[0]}')
print(f'CF rows:   {cf.shape[0]}')

## 1. Revenue distribution — histogram + box plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
rev = pl_latest['sales'].dropna()

axes[0].hist(rev, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Revenue distribution (latest year)')
axes[0].set_xlabel('Sales (₹ Cr)')
axes[0].set_ylabel('Company count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

axes[1].boxplot(rev.values, vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Revenue spread — box plot')
axes[1].set_xlabel('Sales (₹ Cr)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

plt.tight_layout()
plt.show()
print(rev.describe().round(0))

## 2. Top 10 / bottom 10 by revenue

In [ ]:
merged = pl_latest.merge(co[['symbol','company_name','sector']], left_on='company_id', right_on='symbol', how='left')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, df, title, color in [
    (axes[0], merged.nlargest(10, 'sales'), 'Top 10 by revenue', 'steelblue'),
    (axes[1], merged.nsmallest(10, 'sales').sort_values('sales'), 'Bottom 10 by revenue', 'salmon'),
]:
    ax.barh(df['company_id'], df['sales'] / 1000, color=color, alpha=0.85)
    ax.set_xlabel('Sales (₹ 000 Cr)')
    ax.set_title(title)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Top 10 / bottom 10 by net profit, ROE, OPM%

In [ ]:
metrics = [
    ('net_profit',          pl_latest,  'Net profit (₹ Cr)'),
    ('opm_percentage',      pl_latest,  'OPM %'),
]
for col, df_src, label in metrics:
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    for ax, fn, title, c in [
        (axes[0], 'nlargest',  f'Top 10 — {label}',    'steelblue'),
        (axes[1], 'nsmallest', f'Bottom 10 — {label}', 'salmon'),
    ]:
        sub = getattr(df_src, fn)(10, col)
        ax.barh(sub['company_id'], sub[col], color=c, alpha=0.85)
        ax.set_xlabel(label)
        ax.set_title(title)
        ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

# ROE from companies master
roe_df = co[['symbol','roe_percentage']].dropna().sort_values('roe_percentage', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, sub, title, c in [
    (axes[0], roe_df.head(10), 'Top 10 — ROE %', 'steelblue'),
    (axes[1], roe_df.tail(10), 'Bottom 10 — ROE %', 'salmon'),
]:
    ax.barh(sub['symbol'], sub['roe_percentage'], color=c, alpha=0.85)
    ax.set_xlabel('ROE %')
    ax.set_title(title)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Correlation matrix — key financial metrics

In [ ]:
combo = pl_latest[['company_id','sales','net_profit','opm_percentage','net_profit_margin_pct','interest_coverage']].copy()
combo = combo.merge(bs_latest[['company_id','debt_to_equity','equity_ratio']], on='company_id', how='left')
combo = combo.merge(co[['symbol','roe_percentage','roce_percentage']], left_on='company_id', right_on='symbol', how='left')
corr_cols = ['sales','net_profit','opm_percentage','net_profit_margin_pct',
             'interest_coverage','debt_to_equity','equity_ratio','roe_percentage','roce_percentage']
corr = combo[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Correlation matrix — key financial metrics')
plt.tight_layout()
plt.show()
print('\nKey insight: OPM% and net profit margin correlation =', round(corr.loc['opm_percentage','net_profit_margin_pct'], 3))

## 5. Sector-wise aggregations — mean and median of key metrics

In [ ]:
with_sector = pl_latest.merge(co[['symbol','sector']], left_on='company_id', right_on='symbol', how='left')
with_sector_bs = bs_latest.merge(co[['symbol','sector']], left_on='company_id', right_on='symbol', how='left')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, df_s, col, title in [
    (axes[0][0], with_sector,    'opm_percentage',   'Median OPM % by sector'),
    (axes[0][1], with_sector,    'net_profit_margin_pct', 'Median net profit margin by sector'),
    (axes[1][0], with_sector,    'sales',             'Median revenue by sector (₹ Cr)'),
    (axes[1][1], with_sector_bs, 'debt_to_equity',    'Median D/E by sector'),
]:
    agg = df_s.groupby('sector')[col].median().sort_values(ascending=True)
    colors = ['salmon' if v < 0 else 'steelblue' for v in agg.values]
    ax.barh(agg.index, agg.values, color=colors, alpha=0.85)
    ax.set_title(title)
    ax.axvline(0, color='grey', linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Null value heatmap — missing data by company

In [ ]:
null_check = pl.groupby('company_id')[['sales','net_profit','opm_percentage','eps','interest_coverage']].apply(lambda x: x.isnull().mean())
null_check.columns = ['sales','net_profit','opm%','eps','int_cov']
null_check = null_check.sort_values('int_cov', ascending=False)

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(null_check, cmap='Reds', vmin=0, vmax=1,
            linewidths=0.3, cbar_kws={'label': 'Null fraction'}, ax=ax)
ax.set_title('Null fraction by company & metric (P&L)')
ax.set_xlabel('Metric')
ax.set_ylabel('Company')
plt.tight_layout()
plt.show()
print('Companies with >50% nulls in interest_coverage:')
print(null_check[null_check['int_cov'] > 0.5].index.tolist())

## 7. Year coverage per company

In [ ]:
year_count = pl.groupby('company_id')['fiscal_year'].nunique().sort_values()
fig, ax = plt.subplots(figsize=(10, 14))
colors = ['#ef4444' if v < 5 else '#f59e0b' if v < 8 else '#22c55e' for v in year_count.values]
ax.barh(year_count.index, year_count.values, color=colors, alpha=0.9)
ax.set_xlabel('Number of fiscal years')
ax.set_title('Year coverage per company')
ax.axvline(5, color='red', linestyle='--', alpha=0.5, label='5-yr min')
ax.axvline(10, color='green', linestyle='--', alpha=0.5, label='10-yr target')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Companies with <5 years data: {(year_count < 5).sum()}')
print(f'Companies with 10+ years data: {(year_count >= 10).sum()}')

## 8. Distribution of sales growth rates

In [ ]:
pl_sorted = pl.sort_values(['company_id', 'fiscal_year'])
pl_sorted['yoy_growth'] = pl_sorted.groupby('company_id')['sales'].pct_change() * 100
growth_vals = pl_sorted['yoy_growth'].dropna()
growth_vals = growth_vals[(growth_vals > -100) & (growth_vals < 200)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(growth_vals, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0].axvline(growth_vals.median(), color='orange', linestyle='--', alpha=0.9, label=f'Median: {growth_vals.median():.1f}%')
axes[0].set_title('Distribution of YoY sales growth rates')
axes[0].set_xlabel('YoY growth %')
axes[0].legend()

from scipy import stats
_, p_value = stats.normaltest(growth_vals)
print(f'Normality test p-value: {p_value:.4f} (p<0.05 = not normal)')
stats.probplot(growth_vals, plot=axes[1])
axes[1].set_title('Q-Q plot — sales growth rates')
plt.tight_layout()
plt.show()

## 9. Outlier analysis — D/E, OPM%, revenue growth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
outlier_data = [
    (bs_latest['debt_to_equity'].dropna(), 'D/E ratio', axes[0]),
    (pl_latest['opm_percentage'].dropna(), 'OPM %', axes[1]),
    (growth_vals, 'YoY sales growth %', axes[2]),
]
for data, label, ax in outlier_data:
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    outliers = data[(data < q1 - 1.5 * iqr) | (data > q3 + 1.5 * iqr)]
    ax.boxplot(data.values, vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               flierprops=dict(marker='o', color='red', alpha=0.5, markersize=4))
    ax.set_title(f'{label}\n({len(outliers)} outliers)')
    ax.set_ylabel(label)
plt.suptitle('Outlier analysis across key metrics', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 10. Revenue vs net profit scatter by sector

In [ ]:
scatter_df = pl_latest.merge(co[['symbol','sector']], left_on='company_id', right_on='symbol', how='left')
fig, ax = plt.subplots(figsize=(12, 7))
for sector, grp in scatter_df.groupby('sector'):
    ax.scatter(grp['sales']/1000, grp['net_profit']/1000, label=sector, alpha=0.8, s=60)
ax.set_xlabel('Revenue (₹ 000 Cr)')
ax.set_ylabel('Net profit (₹ 000 Cr)')
ax.set_title('Revenue vs net profit by sector (latest year)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 11. Operating margin trend — top 5 IT companies

In [ ]:
it_cos = co[co['sector'] == 'IT']['symbol'].tolist()
it_pl  = pl[pl['company_id'].isin(it_cos) & (pl['fiscal_year'] >= 2015)]
pivot_opm = it_pl.pivot_table(index='fiscal_year', columns='company_id', values='opm_percentage')
fig, ax = plt.subplots(figsize=(12, 5))
for col in pivot_opm.columns:
    ax.plot(pivot_opm.index, pivot_opm[col], marker='o', label=col)
ax.set_title('OPM % trend — IT sector')
ax.set_xlabel('Fiscal year')
ax.set_ylabel('OPM %')
ax.legend()
plt.tight_layout()
plt.show()

## 12. Debt-to-equity distribution — bubble chart by sector

In [ ]:
de_sector = bs_latest.merge(co[['symbol','sector']], left_on='company_id', right_on='symbol', how='left')
sector_de = de_sector.groupby('sector')['debt_to_equity'].agg(['median','count']).reset_index()
sector_de.columns = ['sector','median_de','count']
fig, ax = plt.subplots(figsize=(12, 5))
sc = ax.scatter(range(len(sector_de)), sector_de['median_de'],
                s=sector_de['count']*80, alpha=0.7, c=sector_de['median_de'],
                cmap='RdYlGn_r')
ax.set_xticks(range(len(sector_de)))
ax.set_xticklabels(sector_de['sector'], rotation=45, ha='right')
ax.set_ylabel('Median D/E')
ax.set_title('Median D/E by sector (bubble size = company count)')
plt.colorbar(sc, label='Median D/E')
plt.tight_layout()
plt.show()

## 13. Free cash flow distribution

In [ ]:
fcf = cf_latest['free_cash_flow'].dropna()
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['steelblue' if v >= 0 else 'salmon' for v in fcf.sort_values().values]
ax.bar(range(len(fcf)), fcf.sort_values().values, color=colors, alpha=0.85, width=0.8)
ax.axhline(0, color='black', linewidth=0.7)
ax.set_title('Free cash flow — all companies (latest year), sorted')
ax.set_ylabel('FCF (₹ Cr)')
ax.set_xlabel('Company rank')
plt.tight_layout()
plt.show()
print(f'Positive FCF: {(fcf > 0).sum()} companies')
print(f'Negative FCF: {(fcf < 0).sum()} companies')

## 14. EPS trend — banking sector

In [ ]:
bank_cos = co[co['sector'] == 'Banking']['symbol'].tolist()
bank_pl  = pl[pl['company_id'].isin(bank_cos) & (pl['fiscal_year'] >= 2015)]
pivot_eps = bank_pl.pivot_table(index='fiscal_year', columns='company_id', values='eps')
fig, ax = plt.subplots(figsize=(12, 5))
for col in pivot_eps.columns:
    ax.plot(pivot_eps.index, pivot_eps[col], marker='o', label=col)
ax.set_title('EPS trend — Banking sector')
ax.set_xlabel('Fiscal year')
ax.set_ylabel('EPS (₹)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 15. CAGR analysis — 3Y, 5Y, 10Y periods

In [ ]:
an_pivot = an.pivot_table(index='company_id', columns='period', values='compounded_sales_growth_pct')
if '3Y' in an_pivot.columns and '5Y' in an_pivot.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, period in zip(axes, ['3Y','5Y','10Y']):
        if period not in an_pivot.columns:
            ax.set_visible(False)
            continue
        data = an_pivot[period].dropna().sort_values()
        colors = ['steelblue' if v >= 10 else 'salmon' if v < 0 else '#f59e0b' for v in data.values]
        ax.barh(data.index, data.values, color=colors, alpha=0.85, height=0.7)
        ax.axvline(0, color='grey', linewidth=0.5)
        ax.axvline(10, color='green', linewidth=0.8, linestyle='--', alpha=0.6, label='10% threshold')
        ax.set_title(f'{period} Sales CAGR %')
        ax.set_xlabel('CAGR %')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 16. Interest coverage ratio — risk heatmap

In [ ]:
ic = pl_latest[['company_id','interest_coverage']].dropna().copy()
ic['risk'] = ic['interest_coverage'].apply(lambda x: 'Safe' if x > 5 else 'Watch' if x > 2 else 'High Risk')
fig, ax = plt.subplots(figsize=(12, 4))
colors_map = {'Safe': 'steelblue', 'Watch': '#f59e0b', 'High Risk': 'salmon'}
for risk, grp in ic.groupby('risk'):
    ax.scatter(grp['company_id'], grp['interest_coverage'], label=risk,
               color=colors_map[risk], s=60, alpha=0.85)
ax.axhline(2, color='orange', linestyle='--', alpha=0.7, label='Min threshold (2)')
ax.axhline(5, color='green', linestyle='--', alpha=0.7, label='Safe threshold (5)')
ax.set_title('Interest coverage ratio by company')
ax.set_ylabel('Interest coverage')
ax.set_xticks(range(len(ic)))
ax.set_xticklabels(ic['company_id'], rotation=90, fontsize=7)
ax.legend()
plt.tight_layout()
plt.show()

## 17. Dividend payout % — who rewards shareholders?

In [ ]:
div_avg = pl.groupby('company_id')['dividend_payout'].mean().dropna().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['steelblue' if v >= 30 else '#f59e0b' if v >= 10 else 'salmon' for v in div_avg.values]
ax.bar(div_avg.index, div_avg.values, color=colors, alpha=0.85)
ax.axhline(30, color='green', linestyle='--', alpha=0.6, label='30% payout line')
ax.set_title('Average dividend payout % by company (all years)')
ax.set_ylabel('Avg dividend payout %')
ax.set_xlabel('Company')
plt.xticks(rotation=90, fontsize=7)
ax.legend()
plt.tight_layout()
plt.show()

## 18. Profitability vs leverage quadrant chart

In [ ]:
quad = pl_latest[['company_id','opm_percentage']].merge(
    bs_latest[['company_id','debt_to_equity']], on='company_id', how='inner')
quad = quad.merge(co[['symbol','sector']], left_on='company_id', right_on='symbol', how='left')
quad = quad.dropna(subset=['opm_percentage','debt_to_equity'])

fig, ax = plt.subplots(figsize=(12, 7))
for sector, grp in quad.groupby('sector'):
    ax.scatter(grp['opm_percentage'], grp['debt_to_equity'], label=sector, alpha=0.8, s=80)
med_opm = quad['opm_percentage'].median()
med_de  = quad['debt_to_equity'].median()
ax.axvline(med_opm, color='grey', linestyle='--', alpha=0.5)
ax.axhline(med_de,  color='grey', linestyle='--', alpha=0.5)
ax.set_xlabel('OPM %')
ax.set_ylabel('Debt-to-equity')
ax.set_title('Profitability vs leverage quadrant')
ax.text(med_opm+0.5, med_de+0.2, 'High OPM / High Debt', fontsize=8, color='grey')
ax.text(med_opm+0.5, 0, 'High OPM / Low Debt ✓', fontsize=8, color='green')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 19. Revenue growth heatmap — company × year

In [ ]:
pl_growth = pl.sort_values(['company_id','fiscal_year']).copy()
pl_growth['yoy'] = pl_growth.groupby('company_id')['sales'].pct_change() * 100
pl_recent = pl_growth[pl_growth['fiscal_year'] >= 2016]
pivot = pl_recent.pivot_table(index='company_id', columns='fiscal_year', values='yoy')

fig, ax = plt.subplots(figsize=(16, 18))
sns.heatmap(pivot, cmap='RdYlGn', center=0, vmin=-50, vmax=50,
            linewidths=0.3, linecolor='white', ax=ax,
            cbar_kws={'label': 'YoY revenue growth %'})
ax.set_title('Revenue growth % — company × year heatmap')
ax.set_xlabel('Fiscal year')
ax.set_ylabel('Company')
plt.tight_layout()
plt.show()

## 20. Summary statistics table

In [ ]:
summary = pl_latest[['sales','net_profit','opm_percentage','net_profit_margin_pct','interest_coverage']].describe().round(2)
print('=== P&L Summary Statistics ===')
print(summary)
print()
bs_summary = bs_latest[['borrowings','total_assets','debt_to_equity','equity_ratio']].describe().round(2)
print('=== Balance Sheet Summary ===')
print(bs_summary)
print()
cf_summary = cf_latest[['operating_activity','free_cash_flow']].describe().round(2)
print('=== Cash Flow Summary ===')
print(cf_summary)